# CarSight Feature Expansion

This notebook evaluates whether engine, drivetrain, and body-condition features materially improve the tuned Random Forest. It preserves the production model and writes only a separate v2 candidate artifact.

## 1. Setup and load the processed dataset

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.transformers import EngineValueTransformer

DATA_PATH = PROJECT_ROOT / "data/processed/carsight_clean.csv"
CANDIDATE_PATH = PROJECT_ROOT / "models/random_forest_model_v2_candidate.pkl"
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

Dataset shape: (6674, 28)


## 2. Inspect candidate features

Engine power and displacement contain both exact readings and bands. We inspect types, missingness, and frequent raw formats before defining any transformation.

In [2]:
candidate_features = [
    "motorGucu(HP)", "motorHacmi(Cc)", "cekisTipi",
    "orjinal_parça_sayısı", "lokal_boyalı_parça_sayısı",
    "boyalı_parça_sayısı", "değişen_parça_sayısı",
]
inspection = pd.DataFrame({
    "dtype": df[candidate_features].dtypes.astype(str),
    "missing": df[candidate_features].isna().sum(),
    "unique": df[candidate_features].nunique(dropna=True),
})
inspection

,dtype,missing,unique
motorGucu(HP),str,198,163
motorHacmi(Cc),str,190,153
cekisTipi,str,236,3
orjinal_parça_sayısı,int64,0,14
lokal_boyalı_parça_sayısı,int64,0,12
boyalı_parça_sayısı,int64,0,14
değişen_parça_sayısı,int64,0,10


In [3]:
for column in ["motorGucu(HP)", "motorHacmi(Cc)"]:
    print(f"\n{column} — 20 most frequent formats")
    display(df[column].value_counts(dropna=False).head(20).to_frame("rows"))


motorGucu(HP) — 20 most frequent formats


,rows
motorGucu(HP),
90 hp,588
75 hp,395
110 hp,359
95 hp,312
115 hp,300
100 hp,295
101 - 125 HP,283
105 hp,238
NaN,198



motorHacmi(Cc) — 20 most frequent formats


,rows
motorHacmi(Cc),
1598 cc,968
1401 - 1600 cm3,453
1248 cc,432
1461 cc,407
1560 cc,284
1390 cc,272
1368 cc,198
NaN,190
1201 - 1400 cm3,156


## 3. Preprocessing strategy

- Exact engine readings use the stated number. Closed ranges use their midpoint. A single-bound band such as `1200 cm3'e kadar` uses 1200 as a conservative boundary proxy. Missing or unparseable values remain `NaN`.
- Engine numerics use median imputation learned only from training data. Part-count columns are also median-imputed defensively.
- Categorical columns use an explicit `Eksik` value for missing data and one-hot encoding with unknown categories ignored.
- A fitted Scikit-learn Pipeline stores parsing, imputation, encoding, feature order, and the model together, making single-request inference reproducible.
- Price, listing identifiers, and listing text are excluded to prevent target leakage or memorization. Sparse tax, history, exchange, and seller fields are intentionally excluded from this iteration.

In [4]:
base_features = ["marka", "yıl", "kilometre(Km)", "vitesTipi", "yakitTuru", "kasaTipi"]
expanded_features = base_features + candidate_features
categorical_features = ["marka", "vitesTipi", "yakitTuru", "kasaTipi", "cekisTipi"]
standard_numeric_features = [
    "yıl", "kilometre(Km)", "orjinal_parça_sayısı",
    "lokal_boyalı_parça_sayısı", "boyalı_parça_sayısı", "değişen_parça_sayısı",
]
engine_features = ["motorGucu(HP)", "motorHacmi(Cc)"]

X = df[expanded_features].copy()
y = df["fiyat(TRY)"].copy()
valid_target = y.notna() & np.isfinite(y)
X, y = X.loc[valid_target], y.loc[valid_target]
print(f"Rows available for modeling: {len(X):,}")

Rows available for modeling: 6,674


## 4. Train/test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

Training rows: 5,339; test rows: 1,335


## 5. Build and train the v2 benchmark Pipeline

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Eksik")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
        ("numeric", SimpleImputer(strategy="median"), standard_numeric_features),
        ("engine", Pipeline([
            ("parser", EngineValueTransformer()),
            ("imputer", SimpleImputer(strategy="median")),
        ]), engine_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

v2_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=1, random_state=42, n_jobs=-1,
    )),
])
v2_pipeline.fit(X_train, y_train)
feature_names = v2_pipeline.named_steps["preprocessor"].get_feature_names_out()
print(f"Final transformed input feature count: {len(feature_names):,}")

Final transformed input feature count: 68


## 6. Evaluate MAE, RMSE, and R²

In [7]:
predictions = v2_pipeline.predict(X_test)
v2_metrics = {
    "MAE": mean_absolute_error(y_test, predictions),
    "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
    "R²": r2_score(y_test, predictions),
}
pd.Series(v2_metrics).to_frame("v2 expanded model")

,v2 expanded model
MAE,104918.621150
RMSE,317510.106274
R²,0.858594


## 7. Compare with the current tuned model

In [8]:
old_metrics = {"MAE": 154633.0, "RMSE": 543453.0, "R²": 0.586}
comparison = pd.DataFrame({"current tuned model": old_metrics, "v2 expanded model": v2_metrics})
comparison["absolute change"] = comparison["v2 expanded model"] - comparison["current tuned model"]
comparison["relative change (%)"] = comparison["absolute change"] / comparison["current tuned model"] * 100
comparison

,current tuned model,v2 expanded model,absolute change,relative change (%)
MAE,154633.000,104918.621150,-49714.378850,-32.149916
RMSE,543453.000,317510.106274,-225942.893726,-41.575425
R²,0.586,0.858594,0.272594,46.517761


## 8. Top feature importances

In [9]:
feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": v2_pipeline.named_steps["model"].feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance.head(10)

,feature,importance
0,engine__motorGucu(HP),0.624169
1,numeric__yıl,0.106395
2,numeric__kilometre(Km),0.081654
3,engine__motorHacmi(Cc),0.056148
4,categorical__marka_Rolls-Royce,0.036092
5,categorical__kasaTipi_Coupe,0.029499
6,categorical__vitesTipi_Düz,0.013318
7,categorical__marka_Porsche,0.010311
8,categorical__marka_Audi,0.005256
9,numeric__orjinal_parça_sayısı,0.003087


## 9. Save the candidate and adoption conclusion

In [10]:
mae_improvement = (old_metrics["MAE"] - v2_metrics["MAE"]) / old_metrics["MAE"]
r2_improvement = v2_metrics["R²"] - old_metrics["R²"]
meaningful_improvement = mae_improvement >= 0.05 and r2_improvement >= 0.02

joblib.dump(v2_pipeline, CANDIDATE_PATH)
print(f"Candidate saved to: {CANDIDATE_PATH}")
print(f"MAE improvement: {mae_improvement:.2%}")
print(f"R² improvement: {r2_improvement:+.3f}")
print("Recommendation:", "ADOPT v2" if meaningful_improvement else "REJECT v2 for now")

Candidate saved to: /Users/selcanakturk/Documents/CarSight/models/random_forest_model_v2_candidate.pkl
MAE improvement: 32.15%
R² improvement: +0.273
Recommendation: ADOPT v2


### Decision rule

For this iteration, adoption requires at least a 5% MAE reduction and a 0.02 absolute R² gain versus the reported tuned baseline. The candidate is saved separately for audit and later integration; `random_forest_model.pkl` is never overwritten by this notebook.